# DeepSets vs GraphSAGE Comparison (Extrapolation)

## Research-Backed Methodology

This notebook implements a comprehensive, research-backed comparison between DeepSets (permutation invariant), GraphSAGE (GNN), and SimpleNN (feedforward MLP baseline) models trained on surface code error correction. The comparison follows establishes ML benchmarking practices, strictly replicating the protocol used for NN vs GNN comparisons.

**Key Principles:**
- Fair comparison with parameter budget matching
- Statistical significance testing (Wilcoxon, McNemar's)
- Identical evaluation protocols (Shared Test Sets)
- Multiple performance indices (accuracy, speed, LER, complexity)
- Extrapolation testing (Train on d=3,5,7; Test up to d=13)

**Distances Analyzed:** d=3, d=5, d=7, d=9, d=11, d=13

## 1. Imports and Setup

In [ ]:
import sys
import json
import time
import gc
import os
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

import sklearn
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from scipy import stats
from scipy.stats import wilcoxon, chi2_contingency
import math
import stim

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set paths
BASE_PATH = Path('../..').resolve()  # code/results/deepsets_vs_gnn_comparison -> code/
if str(BASE_PATH) not in sys.path:
    sys.path.insert(0, str(BASE_PATH))

print(f"BASE_PATH: {BASE_PATH}")

# Import custom models
from benchmark_models import DeepSets, SimpleNN, SurfaceCodeSampler
from models import GraphSAGE, SparseGraph

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Directories
RESULTS_DIR = Path("results")
PLOTS_DIR = Path("plots")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_RESULTS_DIR = RESULTS_DIR  # Alias for compatibility

print(f"Outputs will be saved to: {RESULTS_DIR}")

## 2. Configuration

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
# Distances to compare (Extrapolation range)
DISTANCES = [3, 5, 7, 9, 11, 13]

# Model Experiment Name (Extrapolation models trained on d=3,5,7)
EXPERIMENT_NAME = "equal_333333"

# Test dataset configuration
TEST_SAMPLES_PER_DISTANCE = 10000  # Per distance for evaluation
BENCHMARK_SAMPLES = 10000  # For inference speed benchmarking

# Inference benchmarking configuration
BATCH_SIZES = [1, 8, 16, 32, 64, 128, 256]
NUM_WARMUP_RUNS = 10  # Warmup batches before timing
NUM_BENCHMARK_RUNS = 5  # Number of independent runs for statistical robustness
NUM_ACCURACY_RUNS = 4  # Number of independent accuracy evaluations

# Statistical test significance level
ALPHA = 0.05

# Random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Configuration:")
print(f"  Experiment Name: {EXPERIMENT_NAME}")
print(f"  Distances: {DISTANCES}")
print(f"  Test samples per distance: {TEST_SAMPLES_PER_DISTANCE:,}")
print(f"  Batch sizes for benchmarking: {BATCH_SIZES}")

## 3. Helper Functions

In [ ]:
def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model_path: Path) -> float:
    """Get model file size in MB."""
    if model_path.exists():
        return model_path.stat().st_size / (1024 * 1024)
    return 0.0

def load_deepsets_model(experiment_name: str, base_path: Path) -> Tuple[DeepSets, Dict]:
    """Load DeepSets extrapolation model."""
    model_path = base_path / "deepsets" / "extrapolation" / "models" / "revised_training" / f"{experiment_name}.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"DeepSets model not found at {model_path}")
    
    model = DeepSets(base_path=base_path, device=device)
    model.load(str(model_path))
    model.model.eval()

    num_params = count_parameters(model.model)
    model_size_mb = get_model_size_mb(model_path)

    info = {
        'experiment': experiment_name,
        'num_parameters': num_params,
        'model_size_mb': model_size_mb,
        'model_path': str(model_path),
        'config': model._config
    }
    return model, info

def load_graphsage_model(experiment_name: str, base_path: Path) -> Tuple[GraphSAGE, Dict]:
    """Load GraphSAGE extrapolation model."""
    model_path = base_path / "gSAGE" / "extrapolation" / "models" / "revised_training" / f"{experiment_name}.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"GraphSAGE model not found at {model_path}")
    
    model = GraphSAGE(base_path=base_path, device=device)
    model.load(str(model_path))
    model.model.eval()

    num_params = count_parameters(model.model)
    model_size_mb = get_model_size_mb(model_path)

    info = {
        'experiment': experiment_name,
        'num_parameters': num_params,
        'model_size_mb': model_size_mb,
        'model_path': str(model_path),
        'config': getattr(model, 'config', {})
    }
    return model, info

def load_simplenn_model(distance: int, base_path: Path) -> Tuple[SimpleNN, Dict]:
    """Load SimpleNN specialist model for a given distance."""
    model_path = base_path / "nn" / "training" / "models" / "revised_training" / f"d{distance}_specialist.pt"
    
    if not model_path.exists():
        print(f"Warning: SimpleNN model for d={distance} not found.")
        return None, {}
    
    checkpoint = torch.load(model_path, map_location=device)
    num_detectors = checkpoint.get('num_detectors', 0)
    if num_detectors == 0:
         circuit = stim.Circuit.generated("surface_code:rotated_memory_z", rounds=distance, distance=distance, after_clifford_depolarization=0.001)
         num_detectors = circuit.num_detectors

    hidden_dims = checkpoint.get('hyperparams', {}).get('hidden_dims', (512, 1024, 2048))
    dropout = checkpoint.get('hyperparams', {}).get('dropout', 0.1)

    model = SimpleNN(nickname=f"d{distance}", in_channels=num_detectors, hidden_dims=hidden_dims, dropout=dropout, base_path=base_path, device=device)
    try: 
        model.load(str(model_path))
    except: 
        model.model.load_state_dict(checkpoint['state_dict'], strict=False)
    
    model.model.eval()
    num_params = count_parameters(model.model)
    model_size_mb = get_model_size_mb(model_path)

    info = {
        'distance': distance,
        'num_parameters': num_params,
        'model_size_mb': model_size_mb,
        'model_path': str(model_path)
    }
    return model, info

print("Helper functions defined.")

## 4. Load Models

In [ ]:
# Load DeepSets Model (Extrapolation)
print(f"Loading DeepSets Model ({EXPERIMENT_NAME})...")
deepsets_model, ds_info = load_deepsets_model(EXPERIMENT_NAME, BASE_PATH)
print(f"  ✓ DeepSets: {ds_info['num_parameters']:,} params, {ds_info['model_size_mb']:.2f} MB")

# Load GraphSAGE Model (Extrapolation)
print(f"\nLoading GraphSAGE Model ({EXPERIMENT_NAME})...")
graphsage_model, gs_info = load_graphsage_model(EXPERIMENT_NAME, BASE_PATH)
print(f"  ✓ GraphSAGE: {gs_info['num_parameters']:,} params, {gs_info['model_size_mb']:.2f} MB")

# Load SimpleNN Models (Baselines per distance)
print(f"\nLoading SimpleNN Models...")
simplenn_models = {}
simplenn_info = {}
for d in DISTANCES:
    s_model, s_info = load_simplenn_model(d, BASE_PATH)
    if s_model:
        simplenn_models[d] = s_model
        simplenn_info[d] = s_info
        print(f"  ✓ d={d}: {s_info['num_parameters']:,} params")
    else:
        print(f"  ✗ d={d}: Not found")

## 5. Generate Shared Test Sets (Fair Comparison Protocol)

In [ ]:
# Generate shared test datasets
# Includes pre-processing for GraphSAGE (graphs) and DeepSets (coordinates) 
# to ensure benchmarking measures model speed, not data loading speed.
shared_test_data = {}
graph_builder = SparseGraph(device=torch.device('cpu'))
sampler = SurfaceCodeSampler(device=torch.device('cpu'))

print("Generating shared test datasets...")
for d in DISTANCES:
    # Generate detections with fixed seed
    torch.manual_seed(SEED + d)
    np.random.seed(SEED + d)
    
    detections, labels = sampler.sample(
        d=d,
        num_samples=TEST_SAMPLES_PER_DISTANCE,
        p_values=[0.01], # Standard test error rate
        return_p_labels=False
    )
    
    # 1. For GraphSAGE: Convert to PyG Data objects
    # We do this offline to benchmark pure model inference
    graphs = [graph_builder.to_pyg(det, lbl) for det, lbl in zip(detections, labels)]
    
    # 2. For DeepSets: Convert Syndromes to Coordinates
    # DeepSets wrapper handles this, but for fair speed benchmark of the *model*,
    # we pre-calc if we can. However, DeepSets public API is `predict(detections)`. 
    # To be extremely fair to the *architecture* (not the wrapper), we should pre-calc.
    # We'll calculate it using the wrapper's internal method and store it.
    # The forward pass takes (coords, counts).
    ds_detections = detections.to(device)  # Temp move to device for fast conversion
    with torch.no_grad():
        ds_coords, ds_counts = deepsets_model._syndromes_to_coords(ds_detections, d)
    
    shared_test_data[d] = {
        'detections': detections.cpu(),      # For SimpleNN
        'labels': labels.cpu(),
        'graphs': graphs,                    # For GraphSAGE
        'deepsets_input': (ds_coords.cpu(), ds_counts.cpu()), # For DeepSets
        'num_samples': len(labels)
    }
    
    print(f"  ✓ d={d}: {len(labels):,} samples generated & pre-processed")

print(f"\nGenerated shared test data for {len(shared_test_data)} distances")

## 6. Inference Speed Benchmarking

In [ ]:
def benchmark_inference_speed(
    model,
    data,
    batch_sizes: List[int],
    num_warmup: int = 10,
    num_runs: int = 5,
    model_type: str = 'simplenn'
) -> Dict:
    """
    Benchmark inference speed with multiple runs.
    model_type: 'simplenn', 'graph', 'deepsets'
    """
    model.model.eval()
    model.model.to(device)
    
    results = {}
    
    for batch_size in batch_sizes:
        num_samples = len(data) if model_type != 'deepsets' else len(data[0])
        
        # Prepare loader/batches
        if model_type == 'graph':
            loader = DataLoader(data, batch_size=batch_size, shuffle=False)
        elif model_type == 'deepsets':
            # Data is (coords, counts) tuple
            coords, counts = data
            # Create simple batch iterator
            # This is minimal overhead
            pass
        
        # Warmup
        with torch.no_grad():
            warmup_count = 0
            if model_type == 'graph':
                for batch in loader:
                    if warmup_count >= num_warmup: break
                    batch = batch.to(device)
                    _ = model.model(batch)
                    warmup_count += 1
            elif model_type == 'deepsets':
                coords, counts = data
                for i in range(0, min(num_warmup * batch_size, num_samples), batch_size):
                    b_coords = coords[i:i+batch_size].to(device)
                    b_counts = counts[i:i+batch_size].to(device)
                    _ = model.model(b_coords, b_counts)
            else: # simplenn
                 for i in range(0, min(num_warmup * batch_size, num_samples), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    _ = model.model(batch)
        
        if device.type == 'cuda': torch.cuda.synchronize()
        
        # Timed runs
        run_times = []
        for run in range(num_runs):
            if device.type == 'cuda': torch.cuda.synchronize()
            start_time = time.time()
            
            with torch.no_grad():
                if model_type == 'graph':
                    for batch in loader:
                        batch = batch.to(device)
                        _ = model.model(batch)
                elif model_type == 'deepsets':
                    coords, counts = data
                    for i in range(0, num_samples, batch_size):
                        b_coords = coords[i:i+batch_size].to(device)
                        b_counts = counts[i:i+batch_size].to(device)
                        _ = model.model(b_coords, b_counts)
                else:
                    for i in range(0, num_samples, batch_size):
                        batch = data[i:i+batch_size].to(device)
                        _ = model.model(batch)
            
            if device.type == 'cuda': torch.cuda.synchronize()
            run_times.append(time.time() - start_time)
            
        # Calculate statistics
        mean_time = np.mean(run_times)
        results[batch_size] = {
            'mean_time_seconds': mean_time,
            'std_time_seconds': np.std(run_times),
            'throughput_samples_per_sec': num_samples / mean_time,
            'latency_per_sample_us': (mean_time / num_samples) * 1e6
        }
    return results

# Benchmark Models
print("Benchmarking inference speed...")
inference_benchmarks = {}

for d in DISTANCES:
    inference_benchmarks[d] = {}
    
    # 1. DeepSets
    print(f"  d={d}: DeepSets...")
    inference_benchmarks[d]['deepsets'] = benchmark_inference_speed(
        deepsets_model, shared_test_data[d]['deepsets_input'], 
        BATCH_SIZES, NUM_WARMUP_RUNS, NUM_BENCHMARK_RUNS, 'deepsets'
    )
    
    # 2. GraphSAGE
    print(f"  d={d}: GraphSAGE...")
    inference_benchmarks[d]['graphsage'] = benchmark_inference_speed(
        graphsage_model, shared_test_data[d]['graphs'],
        BATCH_SIZES, NUM_WARMUP_RUNS, NUM_BENCHMARK_RUNS, 'graph'
    )
    
    # 3. SimpleNN
    if d in simplenn_models:
        print(f"  d={d}: SimpleNN...")
        inference_benchmarks[d]['simplenn'] = benchmark_inference_speed(
            simplenn_models[d], shared_test_data[d]['detections'],
            BATCH_SIZES, NUM_WARMUP_RUNS, NUM_BENCHMARK_RUNS, 'simplenn'
        )

print("\n✓ Inference benchmarking complete")

## 7. Accuracy Evaluation

In [ ]:
def evaluate_accuracy_metrics(
    model,
    data,
    labels,
    num_runs: int = 4,
    model_type: str = 'simplenn',
    batch_size: int = 256
) -> Dict:
    """Evaluate accuracy with multiple runs."""
    model.model.eval()
    model.model.to(device)
    
    all_accuracies = []
    all_predictions = [] # Store first run for McNemar's
    
    for run in range(num_runs):
        with torch.no_grad():
            predictions = []
            if model_type == 'graph':
                loader = DataLoader(data, batch_size=batch_size, shuffle=False)
                for batch in loader:
                    batch = batch.to(device)
                    predictions.append(model.model(batch).cpu())
            elif model_type == 'deepsets':
                coords, counts = data
                for i in range(0, len(coords), batch_size):
                    b_c = coords[i:i+batch_size].to(device)
                    b_n = counts[i:i+batch_size].to(device)
                    predictions.append(model.model(b_c, b_n).cpu())
            else:
                for i in range(0, len(data), batch_size):
                    batch = data[i:i+batch_size].to(device)
                    predictions.append(model.model(batch).cpu())
            
            predictions = torch.cat(predictions, dim=0).squeeze()
            binary_preds = (predictions >= 0.5).float()
            labels_tensor = labels.float()
            
            acc = (binary_preds == labels_tensor).float().mean().item()
            all_accuracies.append(acc)
            
            if run == 0:
                all_predictions = binary_preds.numpy()
                
    mean_acc = np.mean(all_accuracies)
    std_acc = np.std(all_accuracies)
    ci_95 = 1.96 * std_acc / np.sqrt(num_runs)
    
    return {
        'accuracy_mean': mean_acc,
        'accuracy_std': std_acc,
        'accuracy_ci_95': ci_95,
        'all_accuracies': all_accuracies,
        'predictions': all_predictions
    }

print("Evaluating accuracy metrics...")
accuracy_results = {}

for d in DISTANCES:
    accuracy_results[d] = {}
    labels = shared_test_data[d]['labels']
    
    # DeepSets
    print(f"  d={d}: DeepSets...")
    accuracy_results[d]['deepsets'] = evaluate_accuracy_metrics(
        deepsets_model, shared_test_data[d]['deepsets_input'], labels,
        NUM_ACCURACY_RUNS, 'deepsets'
    )
    
    # GraphSAGE
    print(f"  d={d}: GraphSAGE...")
    accuracy_results[d]['graphsage'] = evaluate_accuracy_metrics(
        graphsage_model, shared_test_data[d]['graphs'], labels,
        NUM_ACCURACY_RUNS, 'graph'
    )
    
    # SimpleNN
    if d in simplenn_models:
        print(f"  d={d}: SimpleNN...")
        accuracy_results[d]['simplenn'] = evaluate_accuracy_metrics(
            simplenn_models[d], shared_test_data[d]['detections'], labels,
            NUM_ACCURACY_RUNS, 'simplenn'
        )

print("\n✓ Accuracy evaluation complete")

## 8. Statistical Tests

In [ ]:
def wilcoxon_test(g1, g2):
    if len(g1) != len(g2): return {'p_value': np.nan}
    stat, p_val = wilcoxon(g1, g2)
    return {'statistic': stat, 'p_value': p_val, 'significant': p_val < ALPHA}

print("Runnning Statistical Tests (DeepSets vs GraphSAGE)...")
stat_results = {}

ds_all_acc = []
gs_all_acc = []

for d in DISTANCES:
    ds_acc = accuracy_results[d]['deepsets']['all_accuracies']
    gs_acc = accuracy_results[d]['graphsage']['all_accuracies']
    
    ds_all_acc.extend(ds_acc)
    gs_all_acc.extend(gs_acc)
    
    stat_results[d] = wilcoxon_test(ds_acc, gs_acc)
    print(f"  d={d}: p-value={stat_results[d]['p_value']:.4f}")

overall_wilcoxon = wilcoxon_test(ds_all_acc, gs_all_acc)
print(f"\nOverall DeepSets vs GraphSAGE p-value: {overall_wilcoxon['p_value']:.6f}")

## 9. Visualization

In [ ]:
# Plot Accuracy vs Distance
plt.figure(figsize=(10, 6))

distances_arr = np.array(DISTANCES)
ds_means = [accuracy_results[d]['deepsets']['accuracy_mean'] for d in DISTANCES]
gs_means = [accuracy_results[d]['graphsage']['accuracy_mean'] for d in DISTANCES]
ds_cis = [accuracy_results[d]['deepsets']['accuracy_ci_95'] for d in DISTANCES]
gs_cis = [accuracy_results[d]['graphsage']['accuracy_ci_95'] for d in DISTANCES]

plt.errorbar(distances_arr, ds_means, yerr=ds_cis, fmt='-o', label='DeepSets', capsize=5)
plt.errorbar(distances_arr, gs_means, yerr=gs_cis, fmt='-s', label='GraphSAGE', capsize=5)

# Add SimpleNN if available
if any('simplenn' in accuracy_results[d] for d in DISTANCES):
    sn_means = [accuracy_results[d]['simplenn']['accuracy_mean'] if d in simplenn_models else np.nan for d in DISTANCES]
    plt.plot(distances_arr, sn_means, '--^', label='SimpleNN (Baseline)', alpha=0.7)

plt.title(f'Accuracy vs Distance ({EXPERIMENT_NAME})')
plt.xlabel('Code Distance (d)')
plt.ylabel('Accuracy')
plt.legend()
plt.xticks(DISTANCES)
plt.savefig(PLOTS_DIR / 'accuracy_comparison.png', dpi=300)
plt.show()

# Plot LER (Log Scale)
plt.figure(figsize=(10, 6))
plt.yscale('log')
ds_ler = [1 - x for x in ds_means]
gs_ler = [1 - x for x in gs_means]
plt.plot(distances_arr, ds_ler, '-o', label='DeepSets')
plt.plot(distances_arr, gs_ler, '-s', label='GraphSAGE')
plt.title('Logical Error Rate vs Distance')
plt.xlabel('Code Distance')
plt.ylabel('Logical Error Rate')
plt.legend()
plt.xticks(DISTANCES)
plt.savefig(PLOTS_DIR / 'ler_comparison.png', dpi=300)
plt.show()

# Plot Inference Speed (Latency at Batch 64)
plt.figure(figsize=(10, 6))
batch_size = 64
ds_latencies = [inference_benchmarks[d]['deepsets'][batch_size]['latency_per_sample_us'] for d in DISTANCES]
gs_latencies = [inference_benchmarks[d]['graphsage'][batch_size]['latency_per_sample_us'] for d in DISTANCES]

plt.plot(distances_arr, ds_latencies, '-o', label='DeepSets')
plt.plot(distances_arr, gs_latencies, '-s', label='GraphSAGE')
plt.title(f'Inference Latency per Sample (Batch {batch_size})')
plt.xlabel('Code Distance')
plt.ylabel('Latency (microseconds)')
plt.legend()
plt.xticks(DISTANCES)
plt.savefig(PLOTS_DIR / 'latency_comparison.png', dpi=300)
plt.show()

## 10. Save Results

In [ ]:
# Save summary results
results_data = {
    'accuracy': accuracy_results,
    'speed': inference_benchmarks,
    'statistics': stat_results,
    'overall_p_value': overall_wilcoxon['p_value']
}

# Helper to convert numpy types for JSON
def convert(o):
    if isinstance(o, np.int64): return int(o)
    if isinstance(o, np.float32): return float(o)
    if isinstance(o, np.float64): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o

with open(RESULTS_DIR / 'comparison_results.json', 'w') as f:
    json.dump(results_data, f, indent=2, default=convert)

print(f"Results saved to {RESULTS_DIR / 'comparison_results.json'}")